# Progressive Disclosure: Controlling What the LLM Sees

Notebook 3 leaned on one visibility rule without unpacking it: prefix an attribute with an underscore and it disappears from the LLM's view. That's one of several levers NOOA gives you for shaping the agent's public surface.

This notebook covers all of them: `@hidden`, leading underscores, `Annotated[T, hidden]`, `@spec(hidden=False)`, and the module-level `with hidden:` block. Whatever you don't hide shows up in `doc(self)`, and `doc(self)` is what the LLM sees.


## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. Replace `"your-api-key"` with a real key for hosted providers; local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
# model = get_llm_client("openai/openai/openai/gpt-5.5", api_key="your-api-key", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)


## A Small Agent

`LibraryAgent` has two public methods (`find_book`, `check_out`), one CodeAct generation method (`help_patron`), and a private helper (`_lookup`).


In [ ]:
import nooa
from nooa import Agent
from nooa.agentdoc import doc

class LibraryAgent(Agent, llm=model):
    """You help patrons find books in a small branch library."""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
        return self._lookup(query)

    def check_out(self, title: str, patron_id: str) -> str:
        """Record a checkout for the given patron and return a confirmation string."""
        return f"Checked out {title!r} to {patron_id}."

    async def help_patron(self, request: str) -> str:
        """Handle a patron request end to end: find the book, then check it out if they want it."""
        ...

    # Underscore prefix → hidden from doc(self). The LLM won't see this method.
    def _lookup(self, query: str) -> str:
        return f"Aisle 3, shelf B ({query})"


## What the LLM Sees

`doc(self)` renders the exact view the LLM gets. `_lookup` is absent.


In [3]:
agent = LibraryAgent()
print(doc(agent))

OTel tracing enabled: journal:http://localhost:5001
class LibraryAgent:
    """You help patrons find books in a small branch library."""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
    def check_out(self, title: str, patron_id: str) -> str:
        """Record a checkout for the given patron and return a confirmation string."""
    async def help_patron(self, request: str) -> str:
        """Handle a patron request end to end: find the book, then check it out if they want it."""


In [6]:
agent = LibraryAgent()
print(doc(agent))

class LibraryAgent:
    """You help patrons find books in a small branch library."""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
    async def help_patron(self, request: str) -> str:
        """Handle a patron request end to end: find the book, then check it out if they want it."""


## The Full Prompt

`print_prompt` renders the entire prompt sent to the LLM for a specific call. The `<self>` block near the end is the class API surface — that's the tool inventory the model chooses from.


In [4]:
await nooa.print_prompt(agent.help_patron, request="I'd like to borrow a book about beekeeping.")

=== SYSTEM PROMPT  [LibraryAgent] ===

<system_prompt expr="self._resolve_system_prompt()">
You help patrons find books in a small branch library.
</system_prompt>

<strategy_prompt>
## Strategy

Jupyter-like Python session. Parameters pre-loaded as locals; state persists across cells. Use `await` directly, `print`/`pprint` to debug, `doc(obj)` to inspect types. You MUST call a tool each turn — **plain-text responses do NOT end the session**. To finish, call `return_result(value)`. Repeated text-only responses will abort the run with an error.

**Your two tools:**
- `execute_python(code)` — run a code cell
- `return_result(value)` — submit your final answer (also callable from inside `execute_python`)

## When to use which tool

Use `return_result(...)` directly for simple answers determinable from the inputs alone (yes/no, one field, a single lookup).

Use `execute_python(...)` for lists/batches, arithmetic, multi-step computation, transforms, or iteration. Always iterate in code — ne

## Hiding a Public Method with `@hidden`

Say `check_out` should only be invoked by your Python code, never chosen by the LLM. Decorate it with `@hidden`. The method still exists and still runs when called directly — it just disappears from `<self>`.


In [5]:
from nooa import hidden

class LibraryAgent(Agent, llm=model):
    """You help patrons find books in a small branch library."""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
        return self._lookup(query)

    @hidden
    def check_out(self, title: str, patron_id: str) -> str:
        """Record a checkout for the given patron and return a confirmation string."""
        return f"Checked out {title!r} to {patron_id}."

    async def help_patron(self, request: str) -> str:
        """Handle a patron request end to end: find the book, then check it out if they want it."""
        ...

    def _lookup(self, query: str) -> str:
        return f"Aisle 3, shelf B ({query})"

agent = LibraryAgent()
await nooa.print_prompt(agent.help_patron, request="I'd like to borrow a book about beekeeping.")

=== SYSTEM PROMPT  [LibraryAgent] ===

<system_prompt expr="self._resolve_system_prompt()">
You help patrons find books in a small branch library.
</system_prompt>

<strategy_prompt>
## Strategy

Jupyter-like Python session. Parameters pre-loaded as locals; state persists across cells. Use `await` directly, `print`/`pprint` to debug, `doc(obj)` to inspect types. You MUST call a tool each turn — **plain-text responses do NOT end the session**. To finish, call `return_result(value)`. Repeated text-only responses will abort the run with an error.

**Your two tools:**
- `execute_python(code)` — run a code cell
- `return_result(value)` — submit your final answer (also callable from inside `execute_python`)

## When to use which tool

Use `return_result(...)` directly for simple answers determinable from the inputs alone (yes/no, one field, a single lookup).

Use `execute_python(...)` for lists/batches, arithmetic, multi-step computation, transforms, or iteration. Always iterate in code — ne

`check_out` is gone from `<self>`. The LLM doesn't know it exists, but `agent.check_out(...)` still works from your Python code.


## The Full Toolbox

Five levers, one rule: **visible by default, hide explicitly** — with the exception of leading underscores, which follow Python's own convention.

| Lever | Where it applies | Effect |
| --- | --- | --- |
| `_leading_underscore` | methods, fields | hidden by default |
| `@hidden` | methods | hide a public method |
| `Annotated[T, hidden]` | fields | hide a field |
| `@spec(hidden=False)` | private methods | expose a `_name` method |
| `with hidden:` | module-level imports | hide imports from the LLM's execution context |

A single class showing four of them:


In [ ]:
from typing import Annotated
from nooa import Agent, hidden, spec

class LibraryAgent(Agent, llm=model):
    """You help patrons find books in a small branch library."""

    # Visible field.
    branch: str = "Main St."

    # Hidden field — secrets, connection handles, anything the LLM should not read.
    api_token: Annotated[str, hidden] = ""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
        return self._lookup(query)

    # Public method, hidden from the LLM.
    @hidden
    def check_out(self, title: str, patron_id: str) -> str:
        """Runs on the harness side only."""
        return f"Checked out {title!r} to {patron_id}."

    # Underscore-prefixed method, explicitly re-exposed.
    @spec(hidden=False)
    def _catalog_stats(self) -> dict:
        """Return catalog stats."""
        return {"total": 12_000, "available": 8_412}

    # Still hidden — no override.
    def _lookup(self, query: str) -> str:
        return f"Aisle 3, shelf B ({query})"

print(doc(LibraryAgent))


`branch` is visible, `api_token` is gone, `check_out` is gone, `_catalog_stats` shows up despite the underscore, and `_lookup` stays hidden.


## Hiding Module-Level Imports

The prompt's `<execution_context>` block lists everything imported in the agent's module — those names are available to the LLM when it writes code. `with hidden:` keeps sensitive or noisy imports out of that list. It's a module-level construct, so it belongs at the top of a `.py` file:

```python
# my_agent.py
from nooa import Agent, hidden

with hidden:
    import boto3
    from mycompany.secrets import load_credentials

class LibraryAgent(Agent):
    ...
```

`boto3` and `load_credentials` are still usable from your methods, but they don't appear in the LLM's execution context.


## Recap

- Everything on `self` is visible to the LLM through `doc(self)` — unless you hide it.
- Underscore-prefix fields and methods for the ordinary Python-convention hiding.
- Use `@hidden` on a public method to keep it Pythonically accessible while removing it from the LLM's view.
- Use `Annotated[T, hidden]` on a field for the same reason (secrets, connection handles, large blobs).
- Use `@spec(hidden=False)` when you want to re-expose an underscore-prefixed method.
- Use `with hidden:` at the top of a module to strip imports from the LLM's execution context.
